# Lesson 71 · Whisper decoding — how do you pick the next word? Greedy decoding and beam search, so Whisper can speak

**Learning goals**
- Understand autoregressive (AR) decoding: at each step, use the previous tokens to predict the next one
- Implement greedy decoding and a simple beam search (width=2)
- Get to know Whisper's system of special tokens

> **In plain words**: greedy = take the top score at every step; beam = keep a few candidates alive at once and compare their totals.

← **Previous lesson** · [L70 · Inside Whisper](L70_whisper_arch.ipynb)

> Last lesson covered **inside Whisper**: log-mel input, the Transformer encoder–decoder and the multitask heads.
> This lesson explores **Whisper decoding**.

## This lesson's story: once Siri has said its first word, how does it keep going?

The last step of speech recognition is turning the encoder's output vectors into text. Whisper doesn't spit out a whole sentence in one go; it is **autoregressive**, "thinking up" the text one word after another, the way people talk.

**The question**: at every step there's the full vocabulary (Whisper has ~51,000 BPE tokens). How do you choose the next word from the probability distribution?

**Two strategies**:

| Strategy | What it does | Analogy |
|---|---|---|
| Greedy decoding | Pick the most probable token at every step | A chess player who always picks the locally best move |
| Beam search (width W) | Track W "candidate sentences" at once, extending all of them at every step | A chess player who plays out W lines at once and keeps those with the best running score |

**Why isn't greedy good enough?**

```
step 1: P("I")=0.6, P("Hi")=0.4         → greedy picks "I"
step 2: P("I love")=0.5, P("I hate")=0.5 → the best greedy path is only 0.6×0.5 = 0.30
step 2: P("Hi there")=0.9               → the "Hi there" path is better: 0.4×0.9 = 0.36!
```

Greedy's "local best" at step 1 can lead to a worse overall result at step 2. Beam search keeps both the "I" and the "Hi" lines alive, and in the end picks the one with the higher total log-probability.

**This lesson's task**: with a toy LM (fixed probability distributions), implement greedy decoding and beam search, and verify numerically how they differ.

In [ ]:
import numpy as np

### Prerequisites: logits, softmax and numerical stability

Before implementing the decoding algorithms, we need three core ideas:

**What are logits?**

A neural network's output layer (a Transformer decoder, say) doesn't output probabilities directly; it outputs **logits** — raw, unprocessed scores. Think of them as:

- **A "rough draft" of probabilities**: logits = [2.5, 0.1, -1.0, 3.2, ...] are the "scores" for different words in the vocabulary
- **An unconstrained range**: logits can be any real number (very large, very small, negative)
- **Mathematically**: logits are usually the raw output of a linear layer applied to an embedding, written $z = W \cdot h + b$

To turn logits into probabilities in [0, 1], you need softmax.

**Why does it have to be softmax?**

Softmax is a "magic" function that turns any list of numbers into a valid probability distribution. It does three things:

1. **Exponentiate** (amplify the differences): $e^{z_i}$ makes big numbers much bigger and small numbers much smaller
2. **Normalize by the sum**: divide by the sum of all the exponentials, so the total probability = 1
3. **Stabilize numerically**: subtract the maximum before computing, to avoid overflow from large numbers

**A demonstration of the problem: why not just normalize the logits directly?**

```
logits = [1000, 1001, 999]
naive normalization: probabilities = [1000/(1000+1001+999), ...]   (that isn't softmax, and breaks for negative logits)

but softmax needs e^1000 = infinity (the computer can't represent it: "overflow")
```

**The fix: subtract the maximum first**

```
logits_normalized = logits - max(logits) = [1000, 1001, 999] - 1001 = [-1, 0, -2]

e^(-1) = 0.368, e^0 = 1, e^(-2) = 0.135 (all finite numbers)

softmax = [0.368, 1, 0.135] / (0.368+1+0.135) ≈ [0.24, 0.67, 0.09]
```

**Key property**: this trick doesn't change the final probabilities — it's mathematically equivalent, just numerically safer.

$\text{softmax}(\mathbf{z}) = \text{softmax}(\mathbf{z} - \max(\mathbf{z}))$

(because the numerator and denominator are both multiplied by $e^{-\max(\mathbf{z})}$, which cancels)


In [ ]:
# Simulate a toy LM: vocabulary size 10, up to 8 decoding steps
np.random.seed(7)
VOCAB_SIZE = 10
EOT = 9            # end-of-text token id (end-of-sentence marker, like the period at the end of a sentence)
MAX_STEPS = 8

def fake_lm(context: list) -> np.ndarray:
    """Toy language model: outputs logits based on the context. (A simulation; in reality this is a Transformer forward pass.)

    Parameters:
      context: the token sequence generated so far (a list), e.g. [0, 1, 2]

    Returns:
      logits: a 1D array of shape (VOCAB_SIZE,), one "raw score" per word

    Design notes:
      - np.random.RandomState(seed) creates a deterministic random generator
        (same seed → same random sequence, which makes the model "deterministically pseudo-random")
      - seed = sum(context) % 137
        - sum(context) adds up the ids of all the context tokens (different contexts usually give different sums)
        - % 137 keeps the sum within [0, 137) (137 is a prime, to avoid too short a cycle)
        - this guarantees: same context → same seed → same logits (reproducible)
      - when len(context) >= 4, the EOT logit is boosted: simulating a real model's "longer sequences tend to stop" behavior
        (a simplification in the toy model that helps show how the decoders differ)

      About the two numbers "4 steps" and "+2.0" — they are arbitrary:
        - They aren't derived from any maths; they're just two "good enough" numbers the author picked,
          so the toy model triggers EOT early on some trajectories and you can watch how
          greedy decoding / beam search handle "stopping early" (instead of always running the full 8 steps).
        - Why 4: MAX_STEPS=8, so taking half the steps guarantees we get a chance to see EOT
          triggered before hitting the step limit; 3 or 5 would work too, without affecting correctness.
        - Why +2.0: the logits are standard-normal samples (roughly between -3 and 3),
          and adding 2.0 very likely squeezes EOT into the high-probability zone after softmax, without making it
          a crushing 100% (otherwise every beam would end neatly at step 4, and you couldn't see that "greedy and beam
          may end at different times"). Switch to +1.0 or +3.0 and EOT just shows up later/earlier,
          less/more certainly — greedy_decode and beam_search don't need a single line changed.
        - In other words: real Whisper has no hard-coded rule like "stop at step 4" —
          when to stop, and how confidently, is decided entirely by the weights the Transformer learned from huge numbers of
          speech–text pairs. These two numbers are only there to fake the effect "EOT will turn up sooner or later"
          in a teaching setting with no real network, so you can focus on the
          decoding algorithms themselves rather than the LM's internals.
    """
    rng = np.random.RandomState(sum(context) % 137)
    logits = rng.randn(VOCAB_SIZE)
    if len(context) >= 4:
        logits[EOT] += 2.0   # after 4 steps it's easier to stop (raises EOT's score; 4 and 2.0 are arbitrary demo parameters, see above)
    return logits

def softmax(logits):
    """Turn logits (any real numbers) into probabilities (in [0,1], summing to 1).

    Key steps:
      1. x = logits - logits.max()  ← numerical stabilization (prevents exp overflow)
      2. e = np.exp(x)               ← exponentiate (amplifies differences)
      3. return e / e.sum()          ← normalize into a probability distribution

    Why subtract logits.max()?
      - if logits = [1000, 1001, 999], np.exp(1000) overflows (far too big)
      - after subtracting the max they become [-1, 0, -2], and np.exp(-1) ≈ 0.368 is finite
      - mathematically equivalent: softmax's ratios don't change; numerator and denominator are both multiplied by e^{-max}, which cancels
      - this is known as the "log-sum-exp stabilization trick"
    """
    x = logits - logits.max()
    e = np.exp(x)
    return e / e.sum()

### Interlude: what is EOT, and where do the numbers "step 4, +2.0" come from?

**First, what EOT is.** EOT stands for end-of-text: a special token in the vocabulary (here its id is 9)
that works like the period at the end of a sentence — when the model generates it, it means "I've finished this sentence; we can stop".
It gets "special treatment" because it doesn't stand for the content of any word; it's a "stop signal":
at every step the decoding loop has to check "is the token we just picked EOT?", and if so it breaks out of the loop and stops generating.

**Next, the two numbers `len(context) >= 4` and `logits[EOT] += 2.0`.**
Reading the code of `fake_lm`, it's natural to ask: "Why 4 steps, not 3 or 5? Why +2.0,
not +1.0 or +3.0? Is there something special about these numbers — would changing them break the algorithm?"

**An analogy**: it's like the "Daniel has 5 apples" in a maths word problem — that "5" has no deeper meaning;
change it to "3 apples" and the skill being tested (addition and subtraction) is exactly the same. The "4" and "+2.0" in `fake_lm`
are the same kind of thing: **props casually set up for a teaching scenario**, not derived from any formula:

- Why **4**: `MAX_STEPS = 8`, so taking half the steps makes it very likely that before actually reaching the step limit (8 steps)
  we'll already have seen EOT triggered once — so you get to see "stopping early",
  instead of dully running the full 8 steps every time. 3, 5 or even 6 would serve the teaching purpose just as well.
- Why **+2.0**: the logits from `rng.randn(...)` fall roughly between -3 and 3,
  and after adding 2.0, EOT's logit is usually the largest or close to the largest of the bunch,
  so after softmax its probability goes up noticeably — but not to 100% (add +100 and EOT would always
  be chosen, greedy decoding and beam search would end neatly at exactly the same step, and you couldn't see
  how the two algorithms differ in "choosing a route" — that would make this exercise pointless).

**The key conclusion (checked in code)**: these two numbers only decide "how quickly, and how confidently, the toy model chooses to stop";
they have **nothing to do** with the algorithmic logic of `greedy_decode` / `beam_search` themselves — swap in any
reasonable numbers and neither decoding function needs a single line changed. The next cell actually runs it,
trying a few different parameter settings to see how the probability of EOT changes.

In [ ]:
### Experiment: swap out "4" and "+2.0" — does the algorithm still run?

# Turn the two numbers hard-coded into fake_lm into parameters, to compare EOT's probability curve under different values.
def fake_lm_variant(context, eot_threshold=4, eot_bump=2.0):
    rng = np.random.RandomState(sum(context) % 137)
    logits = rng.randn(VOCAB_SIZE)
    if len(context) >= eot_threshold:
        logits[EOT] += eot_bump
    return logits

def eot_prob_curve(eot_threshold, eot_bump, n_steps=8):
    """Pretend 0..n_steps-1 tokens have already been generated, and watch how P(EOT) changes at each step."""
    probs_over_time = []
    for step in range(n_steps):
        context = list(range(step))  # the content doesn't matter; only its length simulates "how many steps so far"
        probs = softmax(fake_lm_variant(context, eot_threshold, eot_bump))
        probs_over_time.append(probs[EOT])
    return probs_over_time

print("The lesson's original parameters: eot_threshold=4, eot_bump=2.0")
for step, p in enumerate(eot_prob_curve(eot_threshold=4, eot_bump=2.0)):
    flag = "  ← threshold reached, P(EOT) jumps" if step == 4 else ""
    print(f"  {step} steps generated → P(EOT) = {p:.4f}{flag}")

print()
print("A different setting: eot_threshold=2, eot_bump=1.0 (earlier threshold, smaller bump)")
for step, p in enumerate(eot_prob_curve(eot_threshold=2, eot_bump=1.0)):
    flag = "  ← threshold reached, P(EOT) jumps" if step == 2 else ""
    print(f"  {step} steps generated → P(EOT) = {p:.4f}{flag}")

print()
print("🔑 Key insight:")
print("  1. With different parameters, both when EOT jumps and how far it jumps have changed, but all that has")
print("     changed is 'how keen the toy model is to stop' — the code of greedy_decode() and")
print("     beam_search() hasn't changed at all, and doesn't need to.")
print("  2. This shows that 4 and +2.0 aren't part of the algorithm, just the setup of this exercise;")
print("     swap in any reasonable positive numbers and the decoding algorithms are written exactly the same way.")
print("  3. Real Whisper has no hard-coded rule like \"step 4\" — when EOT should be")
print("     chosen is decided by the weights the Transformer learned; here two numbers just")
print("     fake that 'it'll stop sooner or later' effect so you can focus on the decoding algorithms themselves.")

## ✏️ Task 1: greedy decoding

**What is greedy decoding?**

At every step, pick **the token that is most probable right now** — short-sighted, and fast. Like a chess player who only ever picks the locally strongest move without looking several moves ahead.

**Pseudocode intuition**:
```
sequence = [prompt]
repeat max_steps times:
  1. use the LM to compute the probability distribution for the next step
  2. pick the most probable token
  3. append it to the sequence
  4. if it's EOT (the end token), stop
```

**Why is it called "greedy"?**

Because it **only cares about the immediate payoff** and ignores the long run. In some cases the word that looks most probable at first leads to a worse sentence later (as in the example at the start of this lesson: picking "I" at 0.6 looks best, but the "I …" path ends up scoring lower than "Hi there").

**Signature**:
```python
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    # returns: list[int] — the full sequence including the prompt and the generated tokens (EOT included)
```

**A 3-step implementation**:

| Step | What to do | Details |
|---|---|---|
| 1 | Initialize the sequence as a copy of `prompt` | Don't modify the input |
| 2 | Loop max_steps times: call `fake_lm(seq)` for the logits, softmax, `argmax` to get next_token | O(V) time per step (V = VOCAB_SIZE) |
| 3 | If next_token == eot, append EOT and break; otherwise append and continue | Stops automatically at max_steps (even without generating EOT) |

**Acceptance criteria**:
- The output is a list[int] with length ∈ [len(prompt)+1, len(prompt)+max_steps]
- It ends with the EOT token (or reaches max_steps)
- The result is deterministic (not random) for the seed=7 toy LM

In [ ]:
### Numerical demo: why use log-probabilities?

# Scenario: generate a 100-word sentence where every word has probability 0.1 (a hard task)
# Question 1: what happens if we multiply directly?
prob_per_word = 0.1
sequence_length = 100

# the wrong way: multiply all the probabilities directly
prob_product = prob_per_word ** sequence_length
print(f"❌ Multiplying probabilities: P(sentence) = 0.1^{sequence_length} = {prob_product}")
print(f"   Result: {prob_product} (vanishingly small — a few hundred more words and it underflows to 0)\n")

# the right way: add log-probabilities
log_prob_per_word = np.log(prob_per_word)
log_prob_sum = sequence_length * log_prob_per_word
print(f"✅ Adding log-probabilities: log P(sentence) = {sequence_length} × log(0.1)")
print(f"   = {sequence_length} × {log_prob_per_word:.4f}")
print(f"   = {log_prob_sum:.4f} (a finite number, easy to compare)\n")

# comparison: the log-probabilities of two realistic sentences
print("Scenario: compare two candidate sentences to see which is better")
print("-" * 50)
# sentence A: every word has probability 0.2
log_score_A = 100 * np.log(0.2)
print(f"Sentence A: P(word1)=0.2, P(word2)=0.2, ... × 100 words")
print(f"  log score = 100 × log(0.2) = {log_score_A:.4f}\n")

# sentence B: every word has probability 0.15
log_score_B = 100 * np.log(0.15)
print(f"Sentence B: P(word1)=0.15, P(word2)=0.15, ... × 100 words")
print(f"  log score = 100 × log(0.15) = {log_score_B:.4f}\n")

# compare
print(f"Compare: log_score_A ({log_score_A:.4f}) vs log_score_B ({log_score_B:.4f})")
if log_score_A > log_score_B:
    print(f"✅ Conclusion: sentence A is better (higher score = higher probability)")
else:
    print(f"✅ Conclusion: sentence B is better (higher score = higher probability)")
print()

# key insight
print("🔑 Key insight:")
print("  1. A property of logs: log(a×b) = log(a) + log(b)")
print("     so the product of a long sequence's probabilities becomes a sum of logs (no underflow)")
print("  2. Log is strictly increasing: if a > b, then log(a) > log(b)")
print("     so comparing log values is equivalent to comparing the original probabilities")
print("  3. Log values are usually negative (because probabilities are < 1)")
print("     but that doesn't affect the comparison (-2.3 < -0.105 means P=0.1 < P=0.9)")


In [ ]:
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    """Greedy decoding: pick the most probable token at every step.

    Hints for the algorithm:
      1. Initialize tokens = list(prompt)                # copy the prompt (don't modify the input)
      2. Loop max_steps times:
         a. logits = fake_lm(tokens)                    # the network's raw scores
         b. probs  = softmax(logits)                    # convert to a probability distribution in [0,1], summing to 1
         c. next_token = int(np.argmax(probs))          # find the index of the maximum, convert to int
         d. tokens.append(next_token)                   # add it to the sequence
         e. if next_token == eot: break                 # stop at the end-of-sentence token
      3. Return tokens

    Key points:
      - tokens.append() modifies the list in place (a side effect)
      - np.argmax() returns numpy.int64; convert it with int() before storing it in the list
      - the check (next_token == eot) must come after the append, or the final EOT gets lost
    """
    # ✏️ TODO: implement this function
    raise NotImplementedError("TODO")

prompt = [0, 1, 2]   # simulates [<|startoftranscript|>, <|en|>, <|transcribe|>]
try:
    result = greedy_decode(prompt)
    print(f'Greedy decoding result: {result}')
    # check 1: the prefix is unchanged
    assert result[0:3] == prompt, 'the prefix should be unchanged'
    # check 2: ends with EOT or reaches the step limit (termination condition)
    assert result[-1] == EOT or len(result) - len(prompt) == MAX_STEPS, (
        f'should end with EOT or reach MAX_STEPS={MAX_STEPS} steps; actual sequence: {result}'
    )
    print('✅ Greedy decoding verified')
except (NotImplementedError, TypeError):
    result = None
    print('⚠️  greedy_decode is not implemented yet; result set to None (placeholder)')

## ✏️ Task 2: beam search (width W=2)

**What is beam search?**

Instead of "nailing down" one choice at every step, it **tracks the W most promising candidate sentences at once** (each called a "beam"), extends all of them at every step, and keeps the best W. Analogy: a chess grandmaster playing out several lines at once and finally choosing the one with the best overall score.

**The core data structure of a beam**: each candidate is stored as `(log_prob, sequence)`
```python
# initial state: 1 beam, just the prompt, with log-probability 0 (no new words generated yet)
beams = [(0.0, [0, 1, 2])]
```

**Signature**:
```python
def beam_search(prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    # returns: list[int] — the complete sequence with the highest cumulative log-probability
```

**The 4-step algorithm in detail**:

| Step | What to do | Details |
|---|---|---|
| 1 | **Expand the candidates**: for each beam call `fake_lm(seq)` → softmax → compute the log-probability of all V tokens | W beams × V tokens = **W×V candidates** |
| 2 | **Sort and truncate**: sort by cumulative log-probability, descending, and keep the top W | Keeps the W most promising candidates |
| 3 | **Handle termination**: if a beam ends with EOT, move it into the completed list and stop extending it | Lets different beams finish at different steps |
| 4 | **Return the best**: if beams is empty or everything has finished, take the highest-scoring sequence from completed | Returns the single best sequence (not all W) |

**Why use the cumulative log-probability rather than the single-step probability?**

- **Single-step probability**: only looks at P(token_i) for the current step — it may pick a word that's "likely now but leads somewhere bad"
- **Cumulative probability**: Σ log P(token_1) + log P(token_2) + ... + log P(token_i) — accounts for the quality of the whole path

A high cumulative score = every word in the sentence is reasonably probable = good overall quality.

**A complete example** (W=2, VOCAB_SIZE=3, max_steps=2):

```
start: beams = [(0.0, [0, 1])]

step 1 expansion:
  beam (0.0, [0,1]) expands into 3 candidates
    → (log P(word0), [0,1,0])
    → (log P(word1), [0,1,1])
    → (log P(word2), [0,1,2])
  suppose the log-probabilities are [-1.2, -0.5, -2.0]; keep the top 2:
    beams = [(-0.5, [0,1,1]), (-1.2, [0,1,0])]  ← sorted by score, descending

step 2 expansion:
  beam (-0.5, [0,1,1]) → new scores [-1.7, -1.2, -2.5] (adding -0.5 = cumulative)
  beam (-1.2, [0,1,0]) → new scores [-2.4, -1.9, -3.2] (adding -1.2 = cumulative)
  all 2×3=6 candidates, sorted, keep the top 2:
    beams = [(-1.2, [0,1,1,1]), (-1.7, [0,1,1,0])]
```

**Acceptance criteria**:
- seed=7, W=2: the beam search result **may differ** from the greedy result (showing how the two routes differ)
- The return value is a single sequence (the one with the highest log-probability), not all W

In [ ]:
def beam_search(
    prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT
) -> list:
    """Simple beam search. Returns the highest-scoring sequence.

    Score definition: the sum of all log P(token_i) along the path (higher is better).
      score(seq) = Σ log P(seq[i] | seq[:i])  for i in range(len(prompt), len(seq))

    Why 1e-12?
      - softmax's outputs sum to 1 and in theory shouldn't contain 0
      - but floating-point precision is limited, and very small probabilities can round to 0 (e.g. a very unlikely token)
      - log(0) = -∞ (mathematically meaningless), which breaks the computation
      - 1e-12 is a tiny "safety margin" that makes sure log's input is always positive
      - it doesn't noticeably change the log of a high-probability token (log(0.3 + 1e-12) ≈ log(0.3))
      - but it prevents the log(0) error for low-probability tokens

    Hints for the algorithm:
      1. Initialize beams = [(0.0, list(prompt))]   # (cumulative log-prob, token sequence)
                   completed = []                # finished candidates (ending in EOT)

      2. Loop max_steps times:
         a. candidates = []
         b. for each (score, tokens) in beams:
              logits = fake_lm(tokens)
              probs  = softmax(logits)
              for token_id in range(VOCAB_SIZE):  # ← this runs V times (not width times!)
                  new_score = score + np.log(probs[token_id] + 1e-12)  # cumulative log-probability
                  candidates.append((new_score, tokens + [token_id]))
         c. sort candidates by score, descending (highest first)
            total number of candidates = len(beams) × VOCAB_SIZE (e.g. width=2, V=10 → 20 candidates)
         d. keep the first width candidates as the new beams (e.g. width=2 → keep the first 2)
         e. check whether each new beam ends with EOT:
              - if EOT: move it into completed (this beam is finished, no more extending)
              - if not: keep it in beams (keep extending)
         f. if beams is empty (every candidate ended with EOT): break

      3. Whether the loop broke early (every beam finished) or reached max_steps:
         find the highest-scoring sequence in completed + beams and return it
    """
    # ✏️ TODO: implement this function
    raise NotImplementedError("TODO")

try:
    beam_result = beam_search(prompt, width=2)
    print(f'Beam search result (W=2): {beam_result}')
    if result is not None:
        print(f'Greedy result:            {result}')
    print('(they may differ — beam search explores more paths)')

    # check 1: the return value starts with the prompt
    assert beam_result[:len(prompt)] == prompt, 'beam_result should start with the prompt'

    # check 2: beam search log-prob ≥ greedy log-prob (holds for this toy LM; see the note on optimality below)
    if result is not None:
        def compute_logprob(seq):
            score = 0.0
            tokens = list(prompt)
            for tok in seq[len(prompt):]:
                logits = fake_lm(tokens)
                probs = softmax(logits)
                score += np.log(probs[tok] + 1e-12)
                tokens.append(tok)
            return score
        beam_score   = compute_logprob(beam_result)
        greedy_score = compute_logprob(result)
        assert beam_score >= greedy_score - 1e-6, (
            f'the beam search score ({beam_score:.4f}) should be ≥ the greedy score ({greedy_score:.4f})'
        )
    print('✅ Beam search verified')
except (NotImplementedError, TypeError):
    print('⚠️  beam_search is not implemented yet; skipping the checks')

## 🔗 Aurora connection

How the two decoding algorithms in this lesson map onto the Aurora codebase:

| Implemented in this lesson | Aurora module | Notes |
|---|---|---|
| `greedy_decode` | `aurora.llm.greedy_decode` (`src/aurora/llm/sample.py`) | The src version is a single-step argmax (pick the max of one set of logits); this lesson extends it into a full autoregressive decoding loop |
| `beam_search` | implemented in this notebook | src doesn't include beam search yet; the Whisper paper's long-form transcription setup uses beam_size=5 |

`aurora.llm` currently provides a single-step `greedy_decode` plus `top_k_sample` / `top_p_sample`;
this lesson's greedy loop and beam search show how to assemble "pick one token" into decoding a whole sequence.
L86 builds on this to study temperature / top-k / top-p sampling systematically (`aurora.llm.top_k_sample`).


In [ ]:
### A Python crash course: argmax, list concatenation, comparing logs

# Question 1: what does np.argmax() return?
probs = np.array([0.1, 0.6, 0.2, 0.1])
idx = np.argmax(probs)
print(f"probability array: {probs}")
print(f"np.argmax(probs) = {idx}")
print(f"type: {type(idx)} (numpy int64)")
print(f"int(idx) = {int(idx)} (converted to a Python int)")
print(f"argmax finds the **index** of the maximum (which element it is), not the value itself")
print()

# Question 2: does list concatenation modify the original list?
tokens = [0, 1, 2]
new_token = 3
new_sequence = tokens + [new_token]  # ← this creates a new list
print(f"original list tokens = {tokens}")
print(f"new list tokens + [3] = {new_sequence}")
print(f"was the original list modified? {tokens} (no!)")
print(f"append() modifies the original list, but + creates a new one")
print()

# Question 3: comparing log-probabilities
log_prob_A = -45.3
log_prob_B = -42.1
print(f"log-probabilities of two routes:")
print(f"  route A: log P(seq_A) = {log_prob_A}")
print(f"  route B: log P(seq_B) = {log_prob_B}")
print(f"which is better?")
if log_prob_B > log_prob_A:
    print(f"  ✅ route B is better ({log_prob_B} > {log_prob_A})")
    print(f"  reason: -42.1 is closer to 0, meaning a higher probability")
else:
    print(f"  ✅ route A is better ({log_prob_A} > {log_prob_B})")
print(f"remember: for negative numbers, -2 > -5 (closer to 0 = larger)")
print()

# Question 4: what does the 1e-6 tolerance mean?
score_a = -42.1234567
score_b = -42.1234560
tolerance = 1e-6
print(f"check: beam_score >= greedy_score - 1e-6")
print(f"  beam_score = {score_a:.10f}")
print(f"  greedy_score = {score_b:.10f}")
print(f"  greedy_score - 1e-6 = {score_b - tolerance:.10f}")
print(f"  test: {score_a} >= {score_b - tolerance} ?")
if score_a >= score_b - tolerance:
    print(f"  ✅ passes (tiny floating-point errors allowed)")
else:
    print(f"  ❌ fails")
print(f"why the - 1e-6?")
print(f"  because floating-point arithmetic has rounding error: two values that are equal in theory can differ by 1e-15 to 1e-10")
print(f"  a small tolerance (1e-6) absorbs those rounding errors and avoids false failures")


### How optimal is beam search?

**The question**: does beam search really find the best answer?

**The answer** (on two levels):

1. **Compared with greedy decoding**: ✅ usually, beam search's score ≥ greedy's score
   - Reason: greedy decoding is the special case of beam search with W=1
   - Beam search (W=2) keeps greedy's choice in play alongside another option
   - So Beam(W)'s best route usually scores at least as well as Beam(W=1) = the greedy route — but strictly this isn't guaranteed: greedy's path can be pruned out at a later step if the other beam's expansions look better at that moment and then fade

2. **Compared with the global optimum**: ❌ not necessarily, but usually close
   - Global optimum = go through all V^T possible sequences and pick the highest-scoring one (exponential — infeasible)
   - Beam search (W) only scores about W×V×T candidates (W is the width, usually small)
   - So beam doesn't guarantee the global optimum, but is usually close when W is large enough

   **An intuitive comparison**:
   - Global optimum = a chess engine exhaustively trying every possible line ($30^{100}$ of them — impossible)
   - Beam W=5 = playing out the 5 most promising lines at once (actually feasible, and usually works very well)
   - The larger W, the more likely you are to find the optimum — but the higher the computational cost

**How it shows up in the code**:
```python
# the check verifies "relative optimality" (greedy as a lower bound)
assert beam_score >= greedy_score - 1e-6
# meaning: beam score ≥ greedy score (within floating-point error)
```

This check **can't prove beam search found the global optimum**; it only shows it did at least as well as greedy here (usually better).


## Wrap-up

| Strategy | Complexity | Character |
|---|---|---|
| Greedy | O(T×V) | Fast, locally optimal |
| Beam W=2 | O(T×W×V) | Better, at the cost of W times the computation |
| Top-k / Top-p | O(T×V) | Adds randomness, avoids repetition (see L86) |

At temperature 0 Whisper uses greedy decoding by default; for long-form transcription the paper uses beam_size=5 with a length_penalty to buy higher quality.

Next up, L72: fine-tuning Whisper with LoRA on a real dataset.

## ✏️ Whiteboard challenge: work out the decoding strategies by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Q1**: What is the time complexity of greedy decoding? Let the sequence length be T and the vocabulary size V.
(Each step's argmax is O(V), over T steps, O(T×V) in total)

**Q2**: In beam search (width W), how many candidate sequences are there after each step's expansion? How many are left after taking the top W?
(W×V candidates after expansion; W kept after taking the top W)

**Q3**: If W=1, what does beam search reduce to?
(Greedy decoding — keeping only 1 beam per step is equivalent to argmax)

**Q4**: Why does beam search add log-probabilities instead of multiplying probabilities?
(To avoid floating-point underflow: 0.1^100 ≈ 0, but 100×log(0.1) = -230 is numerically safe; addition is also faster)

**Q5**: Which decoding strategy does Whisper use by default at inference? Why?
(openai-whisper defaults to greedy decoding at temperature 0; the paper's long-form transcription setup uses beam_size=5. Greedy is faster, beam search more accurate — trade speed against quality as needed)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: a demo of greedy's complexity (counting O(T×V) steps)
try:
    import time
    np.random.seed(7)
    prompt_test = [0]
    t0 = time.perf_counter()
    result_greedy = greedy_decode(prompt_test, max_steps=MAX_STEPS)
    t1 = time.perf_counter()
    assert isinstance(result_greedy, list), "greedy_decode should return a list"
    assert result_greedy[0] == 0, "the sequence should start with the prompt"
    # termination condition: ends with EOT, or reaches MAX_STEPS (with this toy LM, starting from [0] doesn't necessarily hit EOT)
    assert result_greedy[-1] == EOT or len(result_greedy) - len(prompt_test) == MAX_STEPS, (
        f"the sequence should end with EOT or reach MAX_STEPS={MAX_STEPS} steps; actual: {result_greedy}"
    )
    print(f"Q1 ✅  greedy_decode: {len(result_greedy)} tokens, O(V={VOCAB_SIZE}) per step, O({len(result_greedy)*VOCAB_SIZE}) in total")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  greedy complexity = O(T×V), T = sequence length, V={VOCAB_SIZE} (to be verified once implemented)")

# Q2: number of beam search candidates
try:
    np.random.seed(7)
    result_beam = beam_search([0], width=2, max_steps=MAX_STEPS)
    assert isinstance(result_beam, list), "beam_search should return a list"
    assert result_beam[0] == 0, "the sequence should start with the prompt"
    W = 2
    print(f"Q2 ✅  Beam W={W}: {W}×{VOCAB_SIZE}={W*VOCAB_SIZE} candidates after each expansion, top-{W} keeps {W} beams")
    # check whether it differs from greedy (showing what beam is worth)
    if result_greedy != result_beam:
        print(f"       ✨ beam result {result_beam} ≠ greedy result {result_greedy} (beam found a better path!)")
    else:
        print(f"       the beam result matches greedy (with a toy LM they're sometimes the same; with a real LM they usually differ)")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  Beam W=2: W×V=2×{VOCAB_SIZE}={2*VOCAB_SIZE} candidates per step, keep the top 2 (to be verified once implemented)")

# Q3: W=1 reduces to greedy
try:
    np.random.seed(7)
    result_w1 = beam_search([0], width=1, max_steps=MAX_STEPS)
    # with W=1, beam should match the greedy result
    assert result_w1 == result_greedy, f"with W=1, beam={result_w1} should equal greedy={result_greedy}"
    print(f"Q3 ✅  W=1 beam search = greedy decode: {result_w1}")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  W=1 reduces to greedy (keeping only 1 beam = argmax) (to be verified once implemented)")
except AssertionError as e:
    print(f"Q3 ⚠️  {e} (check that beam_search(width=1) is implemented correctly)")

# Q4: log-probabilities (numerical check)
probs_q4 = np.array([0.1] * 10)   # a simple uniform distribution
log_probs = np.log(probs_q4 + 1e-10)
assert np.all(np.isfinite(log_probs)), "log_probs should all be finite"
# a 5-step probability product vs a sum of log-probabilities
prob_product = np.prod(probs_q4[:5])
log_sum = np.sum(log_probs[:5])
print(f"Q4 ✅  product of probabilities={prob_product:.2e} (shrinking fast toward underflow), sum of logs={log_sum:.3f} (numerically safe)")

# Q5: Whisper's default strategy (concept check)
print(f"Q5 ✅  Whisper defaults to greedy decoding at temperature 0; the paper's long-form setup uses beam_size=5 (adjustable with --beam_size); greedy = fastest, beam = most accurate")
print("\n🎉 Whisper decoding whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l71_review = {
    "greedy_vs_beam_tradeoff":  None,  # Do you understand the trade-off between greedy's local optimum and beam's wider search? True/False
    "log_prob_rationale":       None,  # Do you understand why log-probabilities are added (to avoid underflow)? True/False
    "greedy_decode_impl":       None,  # Is greedy_decode() correct, with EOT, deterministic for seed=7? True/False
    "beam_search_impl":         None,  # Is beam_search(width=2) correct, returning the sequence with the highest cumulative log-probability? True/False
    "whiteboard_passed":        None,  # Did you finish all 5 whiteboard derivations? True/False
}

unfilled = [k for k, v in l71_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l71_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L71 all clear! On to L72: Fine-tuning Whisper')

---

→ **Next lesson** · [L72 · Fine-tuning Whisper](L72_whisper_finetune.ipynb)

> Next lesson covers **fine-tuning Whisper**: LoRA low-rank adapters vs full fine-tuning, adapting the model to new languages and accents in practice.